# arms · validity — rubric structure, reward-hacking & session shape  `[EVAL]`

**Are the score gains real MI skill or a style shift?** The cross-cutting validity analyses, per arm,
**all four arms on one axis** (PTO_LA0 · PTO_LA5 · GRPO_LA0 · GRPO_LA5): the rubric factor structure
(one global-evaluation halo axis?), the reward-hacking synthesis + its deterministic cross-checks,
session shape (text metrics — no oracle at all), how sessions terminate, the GRPO iter-9 anomaly check,
and ground-truth transcripts. Rendered once per grader → `results/arms/validity/{figures,tables}/<judge>/`.

Global outcomes → `arms/outcomes`; the per-questionnaire decomposition this synthesis draws on (MITI
behaviours, MICI per-behaviour, PCT patient detail, Q2 reward composition) → `arms/questionnaires`;
persona splits → `arms/heterogeneity`; reward faithfulness → `arms/training` + `lookahead/mechanism`;
stats tables → `arms/stats`. The K=0-vs-K=5 *contrast* on these same channels is owned by
`lookahead/behaviour` — this family shows the four arms' *levels* only.

**Arms.** The two K=5 arms stop at their last scored iteration (PTO 10, GRPO **5** — GRPO K=5 is
censored at iteration 5); everything to the right of that on a per-iteration axis is K=0 only, so read
the tail as "K=0 kept training", not as a K difference. Sign conventions are stated per artifact;
`MICI` is lower = better.

In [ ]:
import sys, os
_p = os.path.abspath(".")                      # find eda/ (the dir holding eda_analysis/) from any depth
while _p != os.path.dirname(_p) and not os.path.isdir(os.path.join(_p, "eda_analysis")):
    _p = os.path.dirname(_p)
sys.path.insert(0, _p)
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
pd.set_option("display.width", 185, "display.max_columns", 50)
import eda_analysis
from eda_analysis import exports, plotting, stats, behavior
from eda_analysis.constants import judge_dirname

# ╔═══ FAMILY = which results folder this notebook owns; JUDGE = which grader's scores are read ═══╗
# arms/* is PER-JUDGE: rendered once per grader on disk → results/arms/validity/{figures,tables}/<judge>/.
# render_results.py sets EDA_JUDGE; "" = the primary oracle (gpt-4o-mini), i.e. the numbers the thesis reports.
cfg = eda_analysis.EdaConfig(family="arms/validity", judge=os.environ.get("EDA_JUDGE", ""))
S = eda_analysis.notebook_setup(cfg)
exports.reset_results()   # clears only THIS family's generated figures/tables for the active judge (never SUMMARY.md)
exports.save_provenance(cfg, S.SCORES)   # re-stamp the leaf's _provenance.md (reset just removed the one notebook_setup wrote)

FOCUS = sorted(S.SCORES.arm.unique())                # every arm on disk, one shared axis
GRADER = judge_dirname(S.JUDGE) + (" (primary oracle = the training reward)" if not S.JUDGE else " (held-out judge)")
CENSOR = "GRPO K=5 is censored at iteration 5 (its last scored iteration); PTO K=5 stops at 10 — the tail beyond is K=0 only."
print("grader:", GRADER)

## 1 · Rubric factor structure — two metric families  `[EVAL]`
**Purpose.** Are the rubrics independent skills or one halo? The battery was *designed* as two families, but the data **revises the boundary** — made explicit below in the correlation heatmap (heavy block divider) and the loadings bars (blue vs orange):

- **Global-evaluation (halo) cluster — one factor (blue).** `Q1+Q2`, `WAI-SR`, `CSQ-8`, `MI-SAT`, `MITI` (globals) — subjective satisfaction / alliance / integrity ratings. An **empirical redundancy set, not one official construct** (Q2 and WAI-SR even overlap by design — both alliance measures; per-instrument provenance: `results/METRICS_REFERENCE.md` §1). On their own they collapse to **one factor (PC1≈91%)** — the single-oracle halo — so "all global-eval rubrics rise together" reflects a single latent axis, **not** multi-skill gain.
- **The metrics added on top (orange)** — added to test whether anything loads *off* that factor: **`PCT`** (patient change-talk proportion — the MI *goal*), **`MICI ↓`** (MI-inconsistent / sycophancy rate, lower = better), and the *free* objective MITI-proficiency ratios **`R:Q` / `%CR` / `%MICO`** (derived from existing behavior counts, no rescoring).

**Finding — PCT is not independent of the halo.** Empirically `PCT` (change-talk proportion) loads **with the halo family** (Spearman ρ≈0.79–0.94 under the primary oracle — the held-out leaf's own range is computed into its captions; high PC1 loading): warmer, more satisfying sessions also elicit more patient change-talk, so PCT does not isolate MI *technique*. The **genuine second axis** is **`MICI ↓` + the MITI ratios `R:Q` / `%CR` / `%MICO`** (objective reflection-vs-persuasion technique). **Read:** adding these metrics still drops PC1 from ≈91% to **≈55%** — a real second dimension exists — but it is defined by MI-inconsistency + technique ratios, **not** by PCT. The PCA is **pooled** over every conversation of every arm and iteration; the per-arm PC1 shares are in `rubric_pca_expanded` (the ~91%→~55% drop is *partly mechanical* — appending less-correlated columns lowers the top eigenvalue's share — so read it as "a second dimension exists", not a calibrated effect size). Reward faithfulness lives in `arms/training`.

In [ ]:
# Expand the scores with the FREE objective MITI-proficiency ratios (+ PCT/MICI if already scored).
# (notebook_setup already appended them — add_derived_mitiprof_rows is idempotent, kept for explicitness.)
SCO = eda_analysis.add_derived_mitiprof_rows(S.SCORES, S.ARMS)
EXPANDED = [m for m in (eda_analysis.WARMTH_RUBRICS + eda_analysis.EXTRA_METRICS)
            if m in SCO.questionnaire.unique()]
print("metrics in correlation/PCA:", EXPANDED)

# Inter-rubric correlation over the EXPANDED set (all evaluation metrics), pooled over the four arms.
HALO = [m for m in eda_analysis.WARMTH_RUBRICS if m in SCO.questionnaire.unique()]
_key = [c for c in ["arm", "model", "iteration", "file_index"] if c in SCO.columns]
_wide = SCO.pivot_table(index=_key, columns="questionnaire", values="score", aggfunc="mean")
_rho = _wide[EXPANDED].corr(method="spearman")
PCT_RHO = (f"{_rho.loc['PCT', HALO].min():.2f}-{_rho.loc['PCT', HALO].max():.2f}"
           if "PCT" in _rho.index else "n/a")          # PCT vs the 5 halo rubrics, THIS grader (computed, not quoted)
print("PCT vs halo rubrics Spearman rho range:", PCT_RHO)
fig = plotting.rubric_correlation_heatmap(SCO, metrics=EXPANDED)
exports.save_fig(fig, "rubric_correlation", caption=f"Spearman correlation among all evaluation metrics (per conversation, pooled over every iteration of all four arms; grader {GRADER}). The global-eval (halo) rubrics block together and PCT blocks WITH them (rho {PCT_RHO} vs the 5 halo rubrics under this grader), while MICI + the MITI ratios R:Q/%CR/%MICO form the genuine second family."); plt.show()

# PCA: pooled + per arm. Does adding the further metrics drop the dominant PC1 share?
pca_all = stats.rubric_pca(SCO, metrics=EXPANDED)
PCA_ROWS = []
if pca_all["explained_variance_ratio"]:
    print(f"\nPOOLED: PC1 = {pca_all['explained_variance_ratio'][0]:.1%}  "
          f"(over {len(pca_all['metrics'])} metrics: {pca_all['metrics']})")
    print(f"        PC1 loadings = {pca_all['pc1_loadings']}")
    print("        -> global-eval (halo) rubrics (+ PCT, which loads WITH them) load high on PC1;")
    print("           only MICI + the MITI ratios R:Q/%CR/%MICO stay near-zero = the genuine second axis.")
    PCA_ROWS.append({"arm": "POOLED", "n_metrics": len(pca_all["metrics"]),
                     "PC1_pct": round(100 * pca_all["explained_variance_ratio"][0], 1),
                     "PC2_pct": round(100 * pca_all["explained_variance_ratio"][1], 1) if len(pca_all["explained_variance_ratio"]) > 1 else np.nan,
                     **{f"load_{m}": v for m, v in pca_all["pc1_loadings"].items()}})
for arm in sorted(SCO.arm.unique()):
    p = stats.rubric_pca(SCO[SCO.arm == arm], metrics=EXPANDED)
    if p["explained_variance_ratio"]:
        print(f"{arm}: PC1 = {p['explained_variance_ratio'][0]:.1%}  loadings={p['pc1_loadings']}")
        PCA_ROWS.append({"arm": arm, "n_metrics": len(p["metrics"]),
                         "PC1_pct": round(100 * p["explained_variance_ratio"][0], 1),
                         "PC2_pct": round(100 * p["explained_variance_ratio"][1], 1) if len(p["explained_variance_ratio"]) > 1 else np.nan,
                         **{f"load_{m}": v for m, v in p["pc1_loadings"].items()}})
# The halo family ALONE (the 5 global-eval rubrics) — the ~91% reference the prose quotes.
p_halo = stats.rubric_pca(SCO, metrics=HALO)
if p_halo["explained_variance_ratio"]:
    print(f"\nPOOLED, halo rubrics only ({HALO}): PC1 = {p_halo['explained_variance_ratio'][0]:.1%}")
    PCA_ROWS.append({"arm": "POOLED_halo_only", "n_metrics": len(p_halo["metrics"]),
                     "PC1_pct": round(100 * p_halo["explained_variance_ratio"][0], 1),
                     "PC2_pct": round(100 * p_halo["explained_variance_ratio"][1], 1) if len(p_halo["explained_variance_ratio"]) > 1 else np.nan,
                     **{f"load_{m}": v for m, v in p_halo["pc1_loadings"].items()}})
if PCA_ROWS:
    PCAT = pd.DataFrame(PCA_ROWS)
    display(PCAT)
    exports.save_table(PCAT, "rubric_pca_expanded", caption=f"PCA over the EXPANDED metric set (the 5 global-eval halo rubrics + PCT, MICI and the free MITI ratios R:Q/%CR/%MICO), standardized, per conversation, pooled over every iteration: PC1/PC2 variance share + PC1 loadings, pooled over all four arms and per arm; the `POOLED_halo_only` row is the 5-rubric reference (~91%). Grader {GRADER}. The pooled-vs-halo-only drop is partly mechanical (more, less-correlated columns) — read as 'a second dimension exists', not as an effect size. The default-metric PC1 table lives in arms/stats (rubric_pca_pc1).")

### What PC1 measures — factor loadings  `[EVAL]`
Each metric's loading on the principal components. The 5 global-eval rubrics all load high on **PC1** (~0.44 each) → they are essentially **one** halo factor ('good warm therapist' as a single judgment) — and **`PCT` loads high on PC1 with them** (ρ≈0.79–0.94 under the primary oracle; per-grader range in the caption), so despite the design intent it is *not* a separate axis. Only the objective technique ratios **`R:Q` / `%CR` / `%MICO`** and **`MICI ↓`** load ~0 on PC1 and define **PC2** — they are the axis that measures something the halo doesn't.

In [ ]:
fig = plotting.factor_loadings_bars(SCO, metrics=EXPANDED, components=("PC1", "PC2"))
if fig is not None:
    exports.save_fig(fig, "factor_loadings", caption=f"Each metric's PCA loading (pooled over all four arms and every iteration; grader {GRADER}): the 5 global-eval (halo) rubrics load high on PC1 (one shared factor) and PCT loads high WITH them (Spearman rho {PCT_RHO} vs the halo rubrics under this grader); only the MITI ratios R:Q/%CR/%MICO and MICI load ~0 on PC1 and define PC2."); plt.show()
else:
    print("factor loadings need >=2 metrics with enough rows.")

## 2 · Reward-hacking — the headline concern  `[EVAL]`
The synthesis: the warmth **reward proxy** climbs, but the signals *outside* the reward expose the cost.

**Two confounds this section is built to survive.** (i) **Reward = outcome:** Q1+Q2 is *both* the training reward *and* a headline eval metric — so "Q1+Q2 went up" is partly circular and can't, by itself, prove skill gain. (ii) **Shared oracle:** the simulated patient *and* the primary grader are the **same** model (gpt-4o-mini-2024-07-18), which couples the generator and the evaluator and can inflate the patient-perspective ratings (the held-out judge's leaf of this family, `claude-haiku-4-5/`, is the check on that). The reward-hack conclusion therefore does **not** rest on Q1+Q2. It rests on signals that break these confounds, strongest first:
- **Deterministic text metrics** (turn length ↑, loop %, question rate ↓; `behavior.py` regex — see §3 session shape) — computed from the transcript with **no oracle at all**, so they break **both** confounds. These are the load-bearing evidence.
- **Oracle-scored but *un-rewarded* axes** (`MICI ↑`, `PCT` ~flat, the MITI ratios `R:Q`/`%CR`/`%MICO`) — not part of the reward, so they break the reward=outcome circularity; they still share the oracle *model*, so they corroborate rather than prove.

- **2a** — one twin-axis frame per arm: warmth ↑ **and** MI-inconsistency ↑ *together* while patient change-talk stays ~flat → "all rubrics up" is **not** multi-skill. With four arms the same frame also shows *where* the K=5 arms sit on each channel — the paired K contrast itself is in `lookahead/behaviour`.
- **2b** — question-rate cross-check: regex literal-`?` rate vs the oracle question-*function* count (both /turn, same denominator). They agree on **direction** (questions fall) but **diverge in magnitude** — for GRPO K=0 the regex rate collapses several-fold base→final while the oracle count drops far less (the exact base→final ratios are computed from the table into the figure caption), because late affirmation/advice turns carry question-function without a literal `?`. Syntax vs function, **not a unit bug** (merge is conv-aligned 96/96).
- **2c** — over-praise cross-check: the lexical marker rate agrees directionally with the oracle's `MICI_OverPraiseRate`.

**The per-questionnaire decompositions behind this synthesis live in `arms/questionnaires`:** MITI behaviour drift (B6_AF ↑, B3_Q ↓), MICI per-behaviour (the rise is ~all over-praise), PCT patient detail (globals + talk proportions), and the Q2 item-level reward composition (self-disclosure items top the Δ). (The annotated per-metric curves live in `arms/outcomes/figures/<judge>/trajectories/`; the persona split of the regression in `arms/heterogeneity`.)

In [ ]:
# 2a · The reward-hack in ONE frame — Q1+Q2 reward proxy (left, 1-5) vs MICI/PCT (right, 0-1), twin-axis per arm.
fig = plotting.reward_hack_panel(S.SCORES, arms=FOCUS, palette=S.PALETTE)
if fig is not None:
    exports.save_fig(fig, "reward_hack_panel", caption=f"Twin-axis per arm (all four arms; grader {GRADER}): the global-eval reward proxy Q1+Q2 (left, 1-5) rises while MI-Inconsistency (right, higher=worse) rises with it and Patient Change-Talk (right, the actual MI goal) stays ~flat — 'all rubrics up' is not multi-skill. Per-iteration arm means over 96 personas, no CI band. {CENSOR}")
    exports.save_fig(fig, "reward_hack_panel", group="headline", caption="The reward-hack in one frame — headline copy; canonical: arms/validity/figures/<judge>/reward_hack_panel.png.")
    plt.show()
else:
    print("no focus arms present.")

In [ ]:
# 2b · Question-rate cross-check — deterministic ?-count/turn vs oracle MITI B3_Q/turn (same denominator).
# NOT a unit bug: both are questions-per-therapist-turn, but the numerators are different constructs —
# literal '?' SYNTAX vs oracle question-FUNCTION. They agree on direction (questions fall) but diverge in
# magnitude for GRPO (regex collapses ~7x, oracle ~1.6x): late affirmation/advice turns carry
# question-function the oracle credits without a literal '?'. The widening gap IS the drift signature.
QRC = behavior.question_rate_crosscheck(S.ARMS)
fig = plotting.question_rate_crosscheck(QRC, palette=S.PALETTE)
if fig is not None:
    # base->final fold-change per arm, both ways (computed from the frame so the caption can't go stale)
    def _fold(g, col):
        g = g.sort_values("iteration"); b, f = g[col].iloc[0], g[col].iloc[-1]
        return (b / f) if f else np.nan
    FOLD = {arm: (_fold(g, "q_per_turn"), _fold(g, "q_per_turn_miti")) for arm, g in QRC.groupby("arm")}
    fold_txt = "; ".join(f"{a}: regex {r:.1f}x, oracle {o:.1f}x" for a, (r, o) in sorted(FOLD.items()))
    print("base->final fold-change in questions/turn —", fold_txt)
    exports.save_fig(fig, "question_rate_crosscheck", caption=f"Questions per therapist turn two ways, all four arms: regex literal '?' (solid) vs oracle MITI question-function (B3_Q/turn, dashed; same denominator; oracle side graded by {GRADER}). Both fall, but the regex rate collapses far more than the oracle count (base->final fold-change, base/final: {fold_txt}) — the widening gap = the affirmation/advice drift (declarative prompts carry question-function but no literal '?'). Syntax vs function, not a unit error. {CENSOR}"); plt.show()
    QT = QRC.round(4).sort_values(["arm", "iteration"]).reset_index(drop=True)
    display(QT)
    exports.save_table(QT, "question_rate_crosscheck", caption=f"The frame behind the figure: per (arm, iteration) regex '?'/turn vs oracle MITI B3_Q/turn (grader {GRADER}), conv-aligned means over the 96 conversations. Same denominator (therapist turns); the numerators are syntax vs function.")
else:
    print("MITI not scored yet — question-rate cross-check empty.")

In [ ]:
# 2c · Over-praise cross-check: deterministic lexical marker rate vs the oracle's MICI_OverPraiseRate.
# Validates the DIRECTION of the demoted regex against the professional coder (same role loop% plays
# for degeneration). Empty until the MICI questionnaire is scored via Run_Eval.
XC = behavior.overpraise_crosscheck(S.ARMS)
if XC.empty:
    print("MICI not scored yet — run Run_Eval.ipynb with QUESTIONNAIRE_FILTER=['PCT','MICI'] to populate this cross-check.")
else:
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.scatterplot(XC, x="lex_overpraise_marker_rate", y="MICI_OverPraiseRate",
                    hue="arm", palette=plotting.arm_palette(sorted(XC.arm.unique())), s=60, ax=ax)
    rho = XC[["lex_overpraise_marker_rate", "MICI_OverPraiseRate"]].corr(method="spearman").iloc[0, 1]
    ax.set_title(f"Over-praise: lexical marker vs oracle (Spearman rho={rho:.2f})")
    ax.set_xlabel("lexical over-praise marker rate (regex)"); ax.set_ylabel("oracle MICI over-praise rate")
    plotting.relabel_legend(ax)
    fig.tight_layout()
    exports.save_fig(fig, "overpraise_crosscheck", caption=f"Per-(arm, iteration) deterministic lexical over-praise marker rate vs the oracle-coded MICI over-praise rate (grader {GRADER}), all four arms, pooled Spearman rho in the title — a directional sanity-check on the demoted regex."); plt.show()
    XT = XC.round(4).sort_values(["arm", "iteration"]).reset_index(drop=True)
    display(XT)
    exports.save_table(XT, "overpraise_crosscheck", caption=f"The frame behind the figure: per (arm, iteration) lexical over-praise marker rate (regex) vs oracle MICI_OverPraiseRate (grader {GRADER}); pooled Spearman rho = {rho:.3f} across all (arm, iteration) points.")

## 3 · Session shape — deterministic text metrics  `[EVAL]`
**Purpose.** The no-oracle view of the drift: mean therapist-turn length, regex `?`/turn, degeneration
(loop %) and conversation length across iterations, plus how sessions terminate. These are the
**load-bearing** reward-hack evidence (§2) — computed from the transcripts alone, immune to both the
reward=outcome and shared-oracle confounds, and therefore **identical across the judge leaves** of this
family (only the figure palette/arms differ). **Read:** rising turn length + falling `?`/turn tracks the
advice/affirmation drift; the end-reason mix is a degeneration health-check. Exported:
`session_shape.png` + tables `session_shape_by_iter` / `session_end_reasons`. The persona-paired
K=0-vs-K=5 test on these metrics is `lookahead/behaviour`.

In [ ]:
# 3 · Session shape (deterministic; no oracle) — grid + by-iter table + end-reason mix.
SHAPE = behavior.session_shape_by_iter(S.ARMS)
fig = plotting.behavior_trajectory_grid(
    SHAPE, palette=S.PALETTE, metrics=["mean_turn_len", "q_per_turn", "loop", "conv_len"],
    ncols=2, title="Session shape — deterministic text metrics (no oracle)")
if fig:
    exports.save_fig(fig, "session_shape", caption=f"Deterministic text metrics per (arm, iteration), all four arms: mean therapist-turn length, regex ?/turn, degeneration fraction (loop), conversation length. No oracle involved (judge-invariant by construction) — the load-bearing reward-hack evidence. Per-iteration arm means over the 96 conversations (one point per arm x iteration, no CI band). {CENSOR}"); plt.show()
ST = SHAPE.round(3).sort_values(["arm", "iteration"]).reset_index(drop=True)
display(ST)
exports.save_table(ST, "session_shape_by_iter", caption=f"Mean deterministic text metrics per (arm, iteration), all four arms: turn length (chars), regex ?/turn, loop fraction, conversation length (utterances), therapist turns. No oracle — identical under every judge. {CENSOR}")

# End-reason mix (who/what terminated each session) — degeneration health-check.
rows = []
for a in S.ARMS:
    for k in a.iters:
        cdir = a.conv_dir(k)
        for fn in (os.listdir(cdir) if cdir and os.path.isdir(cdir) else []):
            if fn.startswith("conversation_") and fn.endswith(".csv"):
                try: dd = pd.read_csv(os.path.join(cdir, fn))
                except Exception: continue
                rows.append({"arm": a.label, "ended_by": str(dd["session_ended_by"].iloc[0]) if "session_ended_by" in dd else "NA"})
if rows:
    END = (pd.DataFrame(rows).groupby(["arm", "ended_by"]).size().rename("n").reset_index()
           .pivot_table(index="arm", columns="ended_by", values="n", fill_value=0).reset_index())
    END.columns.name = None
    display(END)
    exports.save_table(END, "session_end_reasons", caption="How sessions terminate per arm, all four arms (count of conversations by session_ended_by, pooled over every scored iteration; `nan` = the session ran to the turn cap with no explicit end) — degeneration health-check. Judge-invariant (read off the transcripts). GRPO K=5 has fewer conversations because it is censored at iteration 5.")

## 4 · GRPO iter-9 anomaly check  `[EVAL]`
**Purpose.** The all-metric trajectory grid (`arms/outcomes`) shows GRPO_LA0 dipping at **iter 9 across
almost every metric simultaneously**, then partially recovering at 10 (while Q1+Q2 declines at both 9 and
10). Persona-paired deltas it8→9, it9→10, it8→10 test whether iter 9 is a one-iteration dip (eval-noise /
transient policy state) vs the start of the regression. **Read:** significant negative it9−it8 followed by
positive it10−it9 = a transient dip; monotonic negative it8→10 on Q1+Q2 = the real reward-hack regression.
(Ported from the old `7_Stats` §6 — it is a per-arm validity check, so it lives here.)

In [ ]:
ARM = "GRPO_LA0"
a = S.SCORES[S.SCORES.arm == ARM]
need = {8, 9, 10}
if not a.empty and need <= set(a.iteration.unique()):
    model_at = lambda it: a[a.iteration == it].model.iloc[0]
    mets = [m for m in ["Q1Q2", "MITI", "WAI-SR"] if m in S.METRICS]
    CHK = pd.concat([stats.compare_two_models(a, model_at(hi), model_at(lo), mets).assign(contrast=f"it{hi}-it{lo}")
                     for lo, hi in [(8, 9), (9, 10), (8, 10)]], ignore_index=True)
    view = CHK[["contrast", "metric", "n", "mean_delta", "dz", "p", "p_holm"]].round(4)
    display(view)
    exports.save_table(view, "grpo_iter9_check",
        caption=f"GRPO_LA0 iter-9 anomaly (grader {GRADER}): persona-paired deltas it8->9, it9->10, it8->10 on Q1+Q2/MITI/WAI-SR (contrast label hi-lo, so + = later iteration higher; N=96 personas paired on persona_id across the per-iteration shuffle). A significant negative it9-it8 followed by a positive it10-it9 = a one-iteration dip (eval-noise or transient policy state), distinct from the monotonic Q1+Q2 reward-hack regression. Holm scope: p_holm is corrected across the 3 metrics WITHIN each contrast.")
else:
    print(f"{ARM} iters 8-10 not all scored — anomaly check skipped.")

## 5 · Persona-matched transcripts  `[EVAL]`
**Purpose.** The same patient persona's conversation early / mid / late per arm — the qualitative drift
in actual words (true-persona recovery makes the match exact across the per-iteration shuffle). Print-only.

In [ ]:
from eda_analysis import persona_order
def file_of_persona(seed, k, pid, n=96): return persona_order(seed, k, n).index(pid)
PERSONA = 0
print("persona", PERSONA, "=", eda_analysis.canonical_personas().loc[PERSONA].to_dict(), "\n")
for arm in S.ARMS:
    iters = arm.iters
    pick = sorted(set([iters[0], iters[len(iters) // 2], iters[-1]]))
    print(f"\n############  {arm.label}  ############")
    for k in pick:
        cdir = arm.conv_dir(k)
        if not cdir: continue
        fi = file_of_persona(arm.seed, k, PERSONA)
        fp = os.path.join(cdir, f"conversation_{fi}.csv")
        if not os.path.exists(fp): continue
        d = pd.read_csv(fp); th = d[d.role == "therapist"]["conversation"].astype(str).tolist()
        print(f"==== {arm.label} model_iter_{k} (conv_{fi}) — {len(th)} therapist turns ====")
        for t in th[1:4]:
            print("   •", " ".join(t.split())[:240]); print()

## 6 · How to read this family
- **Factor structure** (§1): the 5 global-eval rubrics alone share a dominant **PC1** (≈91%) — the single-oracle halo; adding the further metrics drops PC1 to ≈55%, but **PCT loads WITH the halo** (ρ≈0.79–0.94 under the primary oracle; the held-out judge's range is in its leaf's captions) — the genuine second factor is **`R:Q` / `%CR` / `%MICO` + `MICI ↓`** (objective technique + MI-inconsistency). Numbers: `rubric_pca_expanded`.
- **§2 Reward-hacking** is the synthesis, built to survive the reward=outcome and shared-oracle confounds: the load-bearing evidence is the **deterministic** text metrics (§3, no oracle), corroborated by the un-rewarded oracle axes (MICI ↑, PCT flat).
- **§3 session shape** is those deterministic metrics exported: turn length ↑ + `?`/turn ↓ = the drift; loop % + end-reason mix = degeneration health. Judge-invariant — the same table under every `<judge>/` leaf.
- **§4** the GRPO_LA0 iter-9 dip is a one-iteration transient on MITI/WAI-SR but part of the monotonic Q1+Q2 regression.
- **Transcripts** (§5) are the ground truth.
- The per-questionnaire decompositions (MITI behaviour drift + thresholds, MICI per-behaviour ≈ all over-praise, PCT patient detail, Q2 item-level reward composition) live in **`arms/questionnaires`**; the K=0-vs-K=5 *tests* on every channel shown here live in **`lookahead/behaviour`**.
- _(Outcome curves → `arms/outcomes`; persona splits → `arms/heterogeneity`; reward faithfulness → `arms/training`; exact stats → `arms/stats`.)_

## 7 · Artifact index
Refresh `results/arms/INDEX.md` (+ the root `results/INDEX.md`) and drop caption lines whose artifact no longer exists in this family's leaf.

In [ ]:
exports.prune_orphan_captions(); exports.build_index()